# Collectors

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Juntar um stream numa lista, num set, numa String ou num map com collect e Collectors.
- Agrupar e dividir com groupingBy e partitioningBy, e achatar listas aninhadas com flatMap.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. collect: listas, sets e Strings
2. toMap: um valor por chave
3. Agrupando com collectors
4. Agrupando e depois contando ou mapeando
5. partitioningBy: dois grupos
6. flatMap: achatando listas aninhadas

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. collect: listas, sets e Strings

`collect(...)` é a operação terminal que monta um recipiente com os itens do stream. `Collectors.toSet()` tira os repetidos, `Collectors.toCollection(TreeSet::new)` escolhe o tipo de coleção, e `Collectors.joining(", ", "[", "]")` cola Strings com um separador (e um começo e um fim opcionais).

In [ ]:
List<String> tags = List.of("java", "c", "java", "sql", "c");
Set<String> unique = tags.stream().collect(Collectors.toCollection(TreeSet::new));
IO.println(unique);
String line = tags.stream().distinct().collect(Collectors.joining(" | ", "<", ">"));
IO.println(line);

🧪 **Experimente:** junte as tags em maiúsculas, separadas por vírgulas, sem começo nem fim.

### ✏️ Exercício: Números numa linha

Devolva os números juntados por `;`, como `"1;2;3"`. Transforme cada um em String com `map`, depois `Collectors.joining`.

In [ ]:
String joined(List<Integer> values) {
    // TODO: map(String::valueOf) e depois collect(joining(";"))
    return "";
}

void main() {
    check("três números", joined(List.of(1, 2, 3)), "1;2;3");
    check("um", joined(List.of(42)), "42");
    check("nenhum", joined(List.of()), "");
}

<details>
<summary>Ver uma resposta</summary>

```java
String joined(List<Integer> values) {
    return values.stream().map(String::valueOf).collect(Collectors.joining(";"));
}

void main() {
    check("três números", joined(List.of(1, 2, 3)), "1;2;3");
    check("um", joined(List.of(42)), "42");
    check("nenhum", joined(List.of()), "");
}
```

</details>

## 2. toMap: um valor por chave

`Collectors.toMap(funçãoDaChave, funçãoDoValor)` monta um map com uma entrada por item. Dois itens com a mesma chave fazem ele lançar uma `IllegalStateException`, a não ser que um terceiro argumento diga como juntar os valores.

In [ ]:
record Product(String name, double price) {}

void main() {
    List<Product> items = List.of(new Product("pen", 2.5), new Product("book", 30), new Product("pen", 3));
    try {
        items.stream().collect(Collectors.toMap(Product::name, Product::price));
    } catch (IllegalStateException e) {
        IO.println("Duplicate key: " + e.getMessage());
    }
    Map<String, Double> cheapest = items.stream()
            .collect(Collectors.toMap(Product::name, Product::price, Math::min));
    IO.println(cheapest);
}

🧪 **Experimente:** troque a função de junção de `Math::min` para `Double::sum`. O que cada valor significa agora?

## 3. Agrupando com collectors

`collect(Collectors.groupingBy(chave))` monta um **Map** numa linha só: a chave é calculada para cada item, e cada chave recebe a lista dos itens dela. Passe um segundo collector para obter outra coisa por chave, como `Collectors.counting()`.

In [ ]:
List<String> animals = List.of("gato", "galo", "cão", "cabra", "pato", "abelha");
Map<Character, List<String>> byLetter = animals.stream()
        .collect(Collectors.groupingBy(a -> a.charAt(0)));
IO.println(byLetter);
Map<Character, Long> howMany = animals.stream()
        .collect(Collectors.groupingBy(a -> a.charAt(0), Collectors.counting()));
IO.println(howMany);

🧪 **Experimente:** agrupe pela **última** letra. Depois compare com o laço que você precisaria para montar o mesmo mapa na mão.

## 4. Agrupando e depois contando ou mapeando

`groupingBy(chave)` sozinho dá listas de itens inteiros. Um **segundo collector** muda o que cada grupo vira: `counting()` para o tamanho, `mapping(f, toList())` para um campo de cada item, `summingInt(f)` ou `averagingDouble(f)` para um total ou uma média. Os slides acompanham um agrupamento.

#### 🎞️ Agrupando, passo a passo

![groupingBy pega cada cidade, calcula a primeira letra como chave e põe as cidades com a mesma chave na mesma lista; com counting(), cada grupo vira o tamanho dele](grouping-steps.pt-br.svg)

In [ ]:
record Sale(String city, String product, int units) {}

void main() {
    List<Sale> sales = List.of(new Sale("Lima", "tea", 3), new Sale("Rome", "tea", 5),
            new Sale("Lima", "cake", 2), new Sale("Lima", "tea", 1));
    IO.println(sales.stream().collect(Collectors.groupingBy(Sale::city, TreeMap::new, Collectors.counting())));
    IO.println(sales.stream().collect(Collectors.groupingBy(Sale::city, TreeMap::new, Collectors.summingInt(Sale::units))));
    IO.println(sales.stream().collect(Collectors.groupingBy(Sale::city, TreeMap::new,
            Collectors.mapping(Sale::product, Collectors.toSet()))));
}

🧪 **Experimente:** agrupe pelo produto em vez da cidade, e some as unidades.

## 5. partitioningBy: dois grupos

`partitioningBy(teste)` sempre dá um map com exatamente duas chaves, `true` e `false`: os itens que passam no teste e os que não passam. Como o `groupingBy`, ele aceita um segundo collector.

In [ ]:
List<Integer> temps = List.of(31, 18, 25, 35, 12);
Map<Boolean, List<Integer>> hot = temps.stream().collect(Collectors.partitioningBy(t -> t >= 30));
IO.println("quentes: " + hot.get(true) + ", amenas: " + hot.get(false));
IO.println(temps.stream().collect(Collectors.partitioningBy(t -> t >= 30, Collectors.counting())));

🧪 **Experimente:** divida com um teste que ninguém passa (`t > 100`): as duas chaves continuam lá.

### ✏️ Exercício: Pares e ímpares

Divida os números em pares (`true`) e ímpares (`false`) com `partitioningBy`.

In [ ]:
Map<Boolean, List<Integer>> evenOdd(List<Integer> values) {
    // TODO: collect(Collectors.partitioningBy(v -> v % 2 == 0))
    return Map.of();
}

void main() {
    check("misturados", evenOdd(List.of(1, 2, 3, 4)), Map.of(true, List.of(2, 4), false, List.of(1, 3)));
    check("vazio", evenOdd(List.of()), Map.of(true, List.of(), false, List.of()));
}

<details>
<summary>Ver uma resposta</summary>

```java
Map<Boolean, List<Integer>> evenOdd(List<Integer> values) {
    return values.stream().collect(Collectors.partitioningBy(v -> v % 2 == 0));
}

void main() {
    check("misturados", evenOdd(List.of(1, 2, 3, 4)), Map.of(true, List.of(2, 4), false, List.of(1, 3)));
    check("vazio", evenOdd(List.of()), Map.of(true, List.of(), false, List.of()));
}
```

</details>

## 6. flatMap: achatando listas aninhadas

`map` transforma cada item em exatamente um item. `flatMap` transforma cada item num **stream** de itens e junta todos num stream só: linhas em palavras, pedidos nos seus produtos, uma lista de listas numa lista só.

In [ ]:
List<List<String>> orders = List.of(List.of("tea", "cake"), List.of("tea"), List.of("juice", "cake"));
List<String> all = orders.stream().flatMap(List::stream).toList();
IO.println(all);
IO.println(all.stream().distinct().sorted().toList());
List<String> lines = List.of("to be", "or not");
IO.println(lines.stream().flatMap(l -> Arrays.stream(l.split(" "))).toList());

🧪 **Experimente:** troque `flatMap(List::stream)` por `map(List::stream)` e imprima: você ganha uma lista de streams, não de produtos.

## Em resumo

`collect(...)` é a operação terminal que monta um recipiente a partir de um stream, e a classe **Collectors** faz os mais comuns: `joining`, `toSet`, `toMap`, `groupingBy` e `partitioningBy`.

```java
void main() {
    List<String> cities = List.of("Lima", "Lisboa", "Paris", "Porto", "Roma");
    IO.println(cities.stream().collect(Collectors.joining(", ")));
    Map<Character, List<String>> byLetter = cities.stream()
            .collect(Collectors.groupingBy(c -> c.charAt(0)));
    IO.println(byLetter);          // {P=[Paris, Porto], R=[Roma], L=[Lima, Lisboa]}
    Map<Boolean, Long> short4 = cities.stream()
            .collect(Collectors.partitioningBy(c -> c.length() <= 4, Collectors.counting()));
    IO.println(short4);            // {false=3, true=2}

List<List<Integer>> rows = List.of(List.of(1, 2), List.of(3));
    IO.println(rows.stream().flatMap(List::stream).toList());   // [1, 2, 3]
}
```

O `groupingBy` dá um map de cada chave para os itens dela; um segundo collector, como `counting()`, transforma cada grupo em outra coisa.

**Cuidado:** o `toMap` lança uma exceção quando dois itens têm a mesma chave, a não ser que você diga como juntá-los.

## Leituras

- [Using a Collector as a Terminal Operation](https://dev.java/learn/api/collections-and-streams/streams/using-collectors/) (dev.java (Oracle), 20 min): As seções "Collecting in a String of Characters", "Partitioning Elements with a Predicate" e "Collecting in a Map with Grouping By". Pode parar antes das seções de pós-processamento.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Agrupar palavras por tamanho** · Médio
- **Aprovado ou reprovado (partitioningBy)** · Médio
- **Índice de palavras (flatMap)** · Difícil
- Teste: **Relatório da turma** · Difícil